In [8]:
import pandas as pd
import glob

zone_map = {
        "FR": "France",
        "BE": "Belgium",
        "CH": "Switzerland",
        "DE-LU": "Germany",
        "DE-AT-LU": "Germany",
        "DE(TransnetBW)": "Germany",
        "DE(Amprion)": "Germany",
        "ES": "Spain",
        "GB": "Great Britain",
        "GB(ElecLink)": "Great Britain",
        "GB(IFA)": "Great Britain",
        "GB(IFA2)": "Great Britain",
        "IT-North": "Italy-North",
        "IT-North-FR": "Italy-North",
        "IT": "Italy-North",
        "LU": "Germany",
}

hourly_report = pd.read_csv("hourly_report.csv")
hourly_report.shape

(771408, 11)

In [29]:
def load_daily_capacity(df, zone_map):

    # enlever n/e
    df["Capacity (MW)"] = pd.to_numeric(
        df["Capacity (MW)"],
        errors="coerce"
    )

    # extraction zones robuste
    df["from_country"] = (
        df["Out Area"]
        .str.extract(r"\|(.*)")
        .iloc[:, 0]
        .map(zone_map)
    )

    df["to_country"] = (
        df["In Area"]
        .str.extract(r"\|(.*)")
        .iloc[:, 0]
        .map(zone_map)
    )

    # signe directionnel
    df["capacity_mw"] = df.apply(
        lambda r: r["Capacity (MW)"]
        if r["from_country"] == "France"
        else -r["Capacity (MW)"],
        axis=1
    )

    return df[[
        "Day",
        "from_country",
        "to_country",
        "capacity_mw"
    ]]

In [31]:
def fill_capacity_with_daily(hourly_report, daily_capacity):

    df = hourly_report.copy()

    # convertir datetime → jour (datetime64)
    df["day"] = pd.to_datetime(df["datetime"]).dt.normalize()

    # merge CORRECT (clé temporelle incluse)
    df = df.merge(
        daily_capacity,
        left_on=["day", "from_country", "to_country"],
        right_on=["Day", "from_country", "to_country"],
        how="left",
        suffixes=("", "_daily")
    )

    # remplir uniquement les capacités manquantes
    df["capacity_mw"] = df["capacity_mw"].fillna(df["capacity_mw_daily"])

    # nettoyage
    df = df.drop(columns=["Day", "capacity_mw_daily", "day"])

    return df

In [21]:
# dossier contenant les fichiers
path = "data/capacities/daily/*.csv"

# liste des fichiers
files = glob.glob(path)

# lecture + concaténation
daily = pd.concat(
    (pd.read_csv(f) for f in files),
    ignore_index=True
)

# Supprimer la colonne "Time Interval" si elle existe
daily = daily.drop(columns=["Time Interval"])

# Mettre au format datetime
daily["Day"] = pd.to_datetime(
    daily["Day"].str.split(" ").str[1],  # enlève "Friday"
    format="%d/%m/%Y"
)

print(daily.head())

         Day         Out Area In Area Capacity (MW)
0 2021-01-01     BZN|DE-AT-LU  BZN|FR           n/e
1 2021-01-01     BZN|IT-North  BZN|FR         995.0
2 2021-01-01  BZN|IT-North-FR  BZN|FR           n/e
3 2021-01-01        BZN|DE-LU  BZN|FR        1200.0
4 2021-01-01      BZN|GB(IFA)  BZN|FR        2000.0


In [30]:
# main.py
daily_cap = load_daily_capacity(daily, zone_map)
print(daily_cap.head())

         Day   from_country to_country  capacity_mw
0 2021-01-01        Germany     France          NaN
1 2021-01-01    Italy-North     France       -995.0
2 2021-01-01    Italy-North     France          NaN
3 2021-01-01        Germany     France      -1200.0
4 2021-01-01  Great Britain     France      -2000.0


In [ ]:
hourly_report = fill_capacity_with_daily(hourly_report, daily_cap)

# hourly_report.to_csv("hourly_report_filled.csv", index=False)

In [ ]:
# Compter la capacité moyenne pour France → Belgium
mean_capacity = hourly_report[
    (hourly_report["from_country"] == "France") &
    (hourly_report["to_country"] == "Belgium")
]["capacity_mw"].mean()

mean_capacity

1978.0833333333333